# Bakery sales demo: robust model selection with train, validation, and test

This notebook adds safer model training and validation logic for the bakery demand demo.

It includes:
- time-based train / validation / test split
- candidate model comparison
- robust fitting with failure handling
- validation-based model selection
- final scoring on the test set

It also avoids crashing when one mixed model fails to fit.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


Libraries imported successfully.


## 1. Load data


In [4]:
df_raw = pd.read_csv("demo/bakery_kaggle/bakery_sales_revised.csv")
df_raw.head()


,Transaction,Item,date_time,period_day,weekday_weekend
0,1,Bread,10/30/2016 9:58,morning,weekend
1,2,Scandinavian,10/30/2016 10:05,morning,weekend
2,2,Scandinavian,10/30/2016 10:05,morning,weekend
3,3,Hot chocolate,10/30/2016 10:07,morning,weekend
4,3,Jam,10/30/2016 10:07,morning,weekend


In [5]:
print("Shape of raw data:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())
print("\nMissing values:")
print(df_raw.isna().sum())


Shape of raw data: (20507, 5)

Column names:
['Transaction', 'Item', 'date_time', 'period_day', 'weekday_weekend']

Missing values:
Transaction        0
Item               0
date_time          0
period_day         0
weekday_weekend    0
dtype: int64


## 2. Parse datetime and derive time features


In [6]:
df = df_raw.copy()
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df["date"] = df["date_time"].dt.date
df["hour"] = df["date_time"].dt.hour
df["month"] = df["date_time"].dt.month
df["day_name"] = df["date_time"].dt.day_name()

print("Invalid datetimes:", df["date_time"].isna().sum())
df.head()


Invalid datetimes: 0


,Transaction,Item,date_time,period_day,weekday_weekend,date,hour,month,day_name
0,1,Bread,2016-10-30 09:58:00,morning,weekend,2016-10-30,9,10,Sunday
1,2,Scandinavian,2016-10-30 10:05:00,morning,weekend,2016-10-30,10,10,Sunday
2,2,Scandinavian,2016-10-30 10:05:00,morning,weekend,2016-10-30,10,10,Sunday
3,3,Hot chocolate,2016-10-30 10:07:00,morning,weekend,2016-10-30,10,10,Sunday
4,3,Jam,2016-10-30 10:07:00,morning,weekend,2016-10-30,10,10,Sunday


## 3. Aggregate raw transactions into quantity


In [7]:
qty_df = (
    df.groupby(["date", "Item", "period_day", "weekday_weekend"])
      .size()
      .reset_index(name="quantity")
)

qty_df["date"] = pd.to_datetime(qty_df["date"])
qty_df.head()


,date,Item,period_day,weekday_weekend,quantity
0,2016-10-30,Basket,morning,weekend,2
1,2016-10-30,Bread,afternoon,weekend,9
2,2016-10-30,Bread,morning,weekend,20
3,2016-10-30,Cake,afternoon,weekend,1
4,2016-10-30,Chicken sand,afternoon,weekend,1


In [8]:
print("Quantity table shape:", qty_df.shape)
print("Unique items:", qty_df["Item"].nunique())
print(qty_df["quantity"].describe())


Quantity table shape: (5627, 5)
Unique items: 94
count    5627.000000
mean        3.644393
std         4.603694
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        39.000000
Name: quantity, dtype: float64


## 4. Create pseudo prices


In [9]:
base_price_map = {
    "Coffee": 2.8,
    "Bread": 3.5,
    "Tea": 2.5,
    "Cake": 4.8,
    "Pastry": 3.8,
    "Sandwich": 5.5,
    "Medialuna": 2.7,
    "Hot chocolate": 3.2,
    "Cookies": 2.4,
    "Brownie": 3.0,
    "Farm House": 4.5,
    "Muffin": 2.9,
    "Alfajores": 2.6,
    "Juice": 3.0,
    "Soup": 4.9,
    "Scone": 2.8,
    "Toast": 3.2,
    "Scandinavian": 3.8,
    "Truffles": 3.4,
    "Coke": 2.2
}

default_price = 3.5
qty_df["base_price"] = qty_df["Item"].map(base_price_map).fillna(default_price)
qty_df["baseline_revenue"] = qty_df["quantity"] * qty_df["base_price"]
qty_df.head()


,date,Item,period_day,weekday_weekend,quantity,base_price,baseline_revenue
0,2016-10-30,Basket,morning,weekend,2,3.5,7.0
1,2016-10-30,Bread,afternoon,weekend,9,3.5,31.5
2,2016-10-30,Bread,morning,weekend,20,3.5,70.0
3,2016-10-30,Cake,afternoon,weekend,1,4.8,4.8
4,2016-10-30,Chicken sand,afternoon,weekend,1,3.5,3.5


## 5. Prepare categorical variables


In [10]:
qty_df["Item"] = qty_df["Item"].astype("category")
qty_df["period_day"] = qty_df["period_day"].astype("category")
qty_df["weekday_weekend"] = qty_df["weekday_weekend"].astype("category")
qty_df["month"] = qty_df["date"].dt.month.astype("category")

qty_df.dtypes


date                datetime64[s]
Item                     category
period_day               category
weekday_weekend          category
quantity                    int64
base_price                float64
baseline_revenue          float64
month                    category
dtype: object

## 6. Time-based train / validation / test split


In [11]:
qty_df = qty_df.sort_values("date").reset_index(drop=True)

train_cutoff = qty_df["date"].quantile(0.70)
val_cutoff = qty_df["date"].quantile(0.85)

train_df = qty_df[qty_df["date"] <= train_cutoff].copy()
val_df = qty_df[(qty_df["date"] > train_cutoff) & (qty_df["date"] <= val_cutoff)].copy()
test_df = qty_df[qty_df["date"] > val_cutoff].copy()

print("Train size:", train_df.shape)
print("Validation size:", val_df.shape)
print("Test size:", test_df.shape)

print("\nTrain range:", train_df["date"].min(), "→", train_df["date"].max())
print("Validation range:", val_df["date"].min(), "→", val_df["date"].max())
print("Test range:", test_df["date"].min(), "→", test_df["date"].max())


Train size: (3947, 8)
Validation size: (850, 8)
Test size: (830, 8)

Train range: 2016-10-30 00:00:00 → 2017-02-23 00:00:00
Validation range: 2017-02-24 00:00:00 → 2017-03-18 00:00:00
Test range: 2017-03-19 00:00:00 → 2017-04-09 00:00:00


## 7. Remove unused category levels after splitting

This helps reduce singular-matrix problems.


In [12]:
categorical_cols = ["Item", "period_day", "weekday_weekend", "month"]

for frame in [train_df, val_df, test_df]:
    for col in categorical_cols:
        frame[col] = frame[col].cat.remove_unused_categories()

print("Unused categories removed.")


Unused categories removed.


## 8. Define candidate models


In [13]:
candidate_formulas = {
    "Model_A": "quantity ~ period_day + weekday_weekend",
    "Model_B": "quantity ~ period_day * weekday_weekend",
    "Model_C": "quantity ~ period_day + weekday_weekend + month"
}

candidate_formulas


{'Model_A': 'quantity ~ period_day + weekday_weekend',
 'Model_B': 'quantity ~ period_day * weekday_weekend',
 'Model_C': 'quantity ~ period_day + weekday_weekend + month'}

## 9. Train each candidate model on the training set only

This cell tries multiple optimisers and skips failed models.


In [14]:
def try_fit_mixedlm(formula, data, groups):
    methods_to_try = ["lbfgs", "powell", "cg"]
    last_error = None

    for method in methods_to_try:
        try:
            model = smf.mixedlm(formula, data=data, groups=groups)
            result = model.fit(method=method, maxiter=500)
            return result, method, None
        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"

    return None, None, last_error


In [15]:
fitted_models = {}
training_log = []

for model_name, formula in candidate_formulas.items():
    print(f"Training {model_name} with formula: {formula}")

    fitted_result, fit_method, fit_error = try_fit_mixedlm(
        formula=formula,
        data=train_df,
        groups=train_df["Item"]
    )

    if fitted_result is not None:
        fitted_models[model_name] = fitted_result
        training_log.append({
            "model_name": model_name,
            "formula": formula,
            "status": "success",
            "fit_method": fit_method,
            "error": None
        })
        print(f"{model_name} training complete using method: {fit_method}")
    else:
        training_log.append({
            "model_name": model_name,
            "formula": formula,
            "status": "failed",
            "fit_method": None,
            "error": fit_error
        })
        print(f"{model_name} failed.")
        print(f"Reason: {fit_error}")

    print("-" * 70)

training_log_df = pd.DataFrame(training_log)
training_log_df


Training Model_A with formula: quantity ~ period_day + weekday_weekend
Model_A training complete using method: lbfgs
----------------------------------------------------------------------
Training Model_B with formula: quantity ~ period_day * weekday_weekend


/home/xi/miniconda3/envs/pred/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/xi/miniconda3/envs/pred/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/home/xi/miniconda3/envs/pred/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/xi/miniconda3/envs/pred/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
/home/xi/miniconda3/envs/pred/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values

Model_B training complete using method: lbfgs
----------------------------------------------------------------------
Training Model_C with formula: quantity ~ period_day + weekday_weekend + month


/home/xi/miniconda3/envs/pred/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


Model_C training complete using method: lbfgs
----------------------------------------------------------------------


,model_name,formula,status,fit_method,error
0,Model_A,quantity ~ period_day + weekday_weekend,success,lbfgs,None
1,Model_B,quantity ~ period_day * weekday_weekend,success,lbfgs,None
2,Model_C,quantity ~ period_day + weekday_weekend + month,success,lbfgs,None


## 10. Validate all successfully trained models


In [16]:
def score_predictions(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"MSE": mse, "RMSE": rmse, "R2": r2}


In [17]:
validation_rows = []

for model_name, fitted_result in fitted_models.items():
    val_pred = fitted_result.predict(val_df).clip(lower=0)
    metrics = score_predictions(val_df["quantity"], val_pred)

    validation_rows.append({
        "model_name": model_name,
        "formula": candidate_formulas[model_name],
        "val_MSE": metrics["MSE"],
        "val_RMSE": metrics["RMSE"],
        "val_R2": metrics["R2"]
    })

validation_results = pd.DataFrame(validation_rows).sort_values("val_RMSE").reset_index(drop=True)
validation_results


PatsyError: predict requires that you use a DataFrame when predicting from a model
that was created using the formula api.

The original error message returned by patsy is:
mismatching levels: expected ('afternoon', 'evening', 'morning', 'night'), got ('afternoon', 'evening', 'morning')
    quantity ~ period_day + weekday_weekend
               ^^^^^^^^^^

## 11. Select the best validation model


In [ ]:
if validation_results.empty:
    raise ValueError("No candidate model trained successfully, so no validation comparison is available.")

best_model_name = validation_results.loc[0, "model_name"]
best_formula = candidate_formulas[best_model_name]
best_model_result = fitted_models[best_model_name]

print("Best model chosen from validation:")
print("Model name:", best_model_name)
print("Formula:", best_formula)


## 12. Score the chosen model on train, validation, and test


In [ ]:
def evaluate_frame(frame, fitted_result, name):
    preds = fitted_result.predict(frame).clip(lower=0)
    metrics = score_predictions(frame["quantity"], preds)

    print(f"{name} MSE:  {metrics['MSE']:.4f}")
    print(f"{name} RMSE: {metrics['RMSE']:.4f}")
    print(f"{name} R2:   {metrics['R2']:.4f}")
    print("-" * 50)

    return preds, metrics

train_pred_best, train_metrics_best = evaluate_frame(train_df, best_model_result, "TRAIN")
val_pred_best, val_metrics_best = evaluate_frame(val_df, best_model_result, "VALIDATION")
test_pred_best, test_metrics_best = evaluate_frame(test_df, best_model_result, "TEST")


In [ ]:
train_df["pred_quantity"] = train_pred_best
val_df["pred_quantity"] = val_pred_best
test_df["pred_quantity"] = test_pred_best

final_metrics_table = pd.DataFrame([
    {"dataset": "TRAIN", "MSE": train_metrics_best["MSE"], "RMSE": train_metrics_best["RMSE"], "R2": train_metrics_best["R2"]},
    {"dataset": "VALIDATION", "MSE": val_metrics_best["MSE"], "RMSE": val_metrics_best["RMSE"], "R2": val_metrics_best["R2"]},
    {"dataset": "TEST", "MSE": test_metrics_best["MSE"], "RMSE": test_metrics_best["RMSE"], "R2": test_metrics_best["R2"]}
])

final_metrics_table


## 13. Visualise the chosen model on the test set


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(test_df["quantity"], test_df["pred_quantity"], alpha=0.3)
plt.xlabel("Observed quantity (test)")
plt.ylabel("Predicted quantity (test)")
plt.title(f"Chosen model on test set: {best_model_name}")
plt.show()


## 14. Build owner-adjustable price scenarios


In [ ]:
owner_price_adjustment = {
    "Coffee": 1.10,
    "Bread": 1.00,
    "Tea": 0.95,
    "Cake": 1.15,
    "Pastry": 1.05
}

for frame in [train_df, val_df, test_df]:
    frame["adjusted_price"] = frame.apply(
        lambda row: row["base_price"] * owner_price_adjustment.get(row["Item"], 1.0),
        axis=1
    )


In [ ]:
for frame in [train_df, val_df, test_df]:
    frame["pred_revenue_base"] = frame["pred_quantity"] * frame["base_price"]
    frame["pred_revenue_adjusted"] = frame["pred_quantity"] * frame["adjusted_price"]

test_df.head()


## 15. Revenue comparison on the test set


In [ ]:
test_revenue_compare = (
    test_df.groupby("Item", observed=False)[["pred_revenue_base", "pred_revenue_adjusted"]]
    .sum()
    .sort_values("pred_revenue_base", ascending=False)
)

test_revenue_compare.head(15)


In [ ]:
top_rev_items = test_revenue_compare.head(10)
top_rev_items.plot(kind="bar", figsize=(12, 5))
plt.title("Test set: top 10 items by baseline vs adjusted predicted revenue")
plt.ylabel("Predicted revenue")
plt.xlabel("Item")
plt.xticks(rotation=45, ha="right")
plt.show()


## 16. Daily forecast table from the test set


In [ ]:
daily_test_forecast = (
    test_df.groupby("date", observed=False)[["pred_revenue_base", "pred_revenue_adjusted"]]
    .sum()
    .reset_index()
    .sort_values("date")
)

daily_test_forecast.head()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily_test_forecast["date"], daily_test_forecast["pred_revenue_base"], label="Baseline")
plt.plot(daily_test_forecast["date"], daily_test_forecast["pred_revenue_adjusted"], label="Adjusted")
plt.title("Test set: daily predicted revenue")
plt.ylabel("Revenue")
plt.xlabel("Date")
plt.legend()
plt.xticks(rotation=45)
plt.show()


## 17. Export outputs


In [ ]:
train_df.to_csv("demo/bakery_kaggle/bakery_train_predictions.csv", index=False)
val_df.to_csv("demo/bakery_kaggle/bakery_validation_predictions.csv", index=False)
test_df.to_csv("demo/bakery_kaggle/bakery_test_predictions.csv", index=False)

daily_test_forecast.to_csv("demo/bakery_kaggle/bakery_test_daily_revenue_forecast.csv", index=False)
training_log_df.to_csv("demo/bakery_kaggle/bakery_training_log.csv", index=False)
validation_results.to_csv("demo/bakery_kaggle/bakery_validation_model_comparison.csv", index=False)
final_metrics_table.to_csv("demo/bakery_kaggle/bakery_final_metrics_table.csv", index=False)

print("Export complete.")


In [ ]:
import os
print("Current working directory:")
print(os.getcwd())
